# Phase 3 - Step 2: Vessel Segmentation (8 TransUNet)

## What is TransUNet?
TransUNet (Chen et al., 2021) is a **hybrid** architecture that combines the strengths of both CNNs and Transformers.

### The Best of Both Worlds
- **CNNs** are great at capturing local features (edges, textures) but struggle with long-range dependencies
- **Transformers** excel at global relationships (self-attention) but are poor at fine-grained local details
- **TransUNet** uses a CNN for the first part of encoding (local features), then passes those features through a ViT (global relationships), then uses a CNN decoder with skip connections

### Architecture Flow
```
Image (3, 512, 512)
  │
  ▼ CNN Encoder (ResNet50 first 3 stages)
Feature Maps (1024, 32, 32)  ← Captures local features
  │
  ▼ Reshape to sequence of patches
Patch Sequence (1024 tokens, dim=768)
  │
  ▼ Transformer Encoder (12 layers of Multi-Head Self-Attention)
Contextualized Tokens  ← Every patch now "knows" about every other patch
  │
  ▼ Reshape back to 2D feature map
Feature Map (768, 32, 32)
  │
  ▼ CNN Decoder + Skip Connections from CNN Encoder
Output (1, 512, 512)
```

### Why Hybrid Helps for Vessels
- CNN captures fine vessel edges and textures (local)
- Transformer captures the vessel tree structure and connectivity (global)
- Skip connections preserve spatial precision for thin vessels

## Training Strategy: Transfer Learning
- ResNet50 encoder: pretrained on ImageNet
- ViT: pretrained on ImageNet (via timm)
- Decoder: trained from scratch

In [ ]:
!pip install -q albumentations timm



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import timm
from torchvision import models



In [ ]:
# Our shared modules sice we are going to provide them in github as moduls and python functions we will write the here

# 1 from src.dataset import RetinalDataset, get_train_transforms, get_val_transforms

"""
Retinal DR Detection - PyTorch Dataset & Augmentation Pipelines
================================================================
This module provides the RetinalDataset class that loads preprocessed
fundus images and their corresponding vessel/lesion masks.

It uses Albumentations for on-the-fly augmentation, ensuring that
geometric transforms (flip, rotate, distort) are applied identically
to both the image and its mask — critical for segmentation tasks.
"""

import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


class RetinalDataset(Dataset):
    """
    PyTorch Dataset for retinal fundus images and segmentation masks.

    This dataset reads image-mask pairs from a CSV file generated by
    the preprocessing notebook (01_preprocessing.ipynb). Each row in
    the CSV contains relative paths to the processed image and its
    corresponding vessel mask.

    Args:
        csv_file (str): Path to the CSV split file (train/val/test).
        base_dir (str): Root directory of the processed dataset.
        transform (callable): Albumentations transform pipeline.
        mask_col (str): Column name in CSV pointing to the mask path.
                        Default 'vessel_path' for vessel segmentation.
    """

    def __init__(self, csv_file, base_dir, transform=None, mask_col='vessel_path'):
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform
        self.mask_col = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build full paths
        img_path = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row[self.mask_col])

        # Read image (BGR -> RGB for Albumentations)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read mask (grayscale, binary)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  # Binarize to 0.0 / 1.0

        # Apply augmentations (same transform applied to BOTH image and mask)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Ensure mask has channel dimension: (1, H, W)
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ============================================================
# Augmentation Pipelines
# ============================================================

def get_train_transforms(img_size=512):
    """
    Training augmentation pipeline.

    Geometric transforms are critical for retinal images because:
    - The retina can be photographed from any angle -> flips & rotations
    - Different cameras have different FOVs -> scale variations
    - Slight patient movement -> shift/distortion

    Color transforms handle:
    - Different fundus camera brands (Topcon vs Zeiss vs Canon)
    - Different illumination conditions
    """
    return A.Compose([
        A.Resize(img_size, img_size),

        # === Geometric Augmentations ===
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625,
            scale_limit=0.1,
            rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),

        # === Color / Illumination Augmentations ===
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.2),

        # === Normalization (ImageNet stats for pretrained backbones) ===
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=512):
    """
    Validation/Test transform — NO augmentation, only resize + normalize.
    We must evaluate on clean, unaugmented images for fair comparison.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# 2 from src.metrics import evaluate_full
"""
Retinal DR Detection - Medical Evaluation Metrics
===================================================
Comprehensive metrics for evaluating segmentation quality in clinical context.

In medical imaging, standard accuracy is NOT enough. A model that predicts
"no vessel everywhere" would get ~90% accuracy (because vessels are only ~10%
of the image), but would be completely useless clinically.

Key metrics explained:
- Dice Coefficient: Overlap between prediction and ground truth (harmonic mean of P & R)
- IoU (Jaccard): Intersection over Union — stricter than Dice
- Sensitivity (Recall): "Of all actual vessels, how many did we detect?"
  → Critical in medicine: missing a vessel = missing pathology
- Specificity: "Of all non-vessel pixels, how many did we correctly identify?"
  → Important to avoid false alarms
- Precision (PPV): "Of everything we predicted as vessel, how much was correct?"
- AUC-ROC: Overall discriminative ability across all thresholds
- AUC-PR: Better than AUC-ROC for imbalanced data (which vessel segmentation is)
"""

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix


# ============================================================
# Pixel-Level Metrics (operate on binary masks)
# ============================================================

def dice_coefficient(pred, target, smooth=1e-6):
    """
    Dice = 2 * |P ∩ T| / (|P| + |T|)

    Ranges from 0 (no overlap) to 1 (perfect overlap).
    This is THE standard metric for medical image segmentation.

    Args:
        pred: Binary prediction tensor (B, 1, H, W)
        target: Binary ground truth tensor (B, 1, H, W)
        smooth: Smoothing factor to avoid division by zero
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)


def iou_score(pred, target, smooth=1e-6):
    """
    IoU (Jaccard Index) = |P ∩ T| / |P ∪ T|

    Stricter than Dice — penalizes false positives and false negatives more.
    IoU of 0.5 means the prediction overlaps only half the ground truth area.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)


def sensitivity(pred, target, smooth=1e-6):
    """
    Sensitivity (Recall / True Positive Rate)
    = TP / (TP + FN)

    "Of all real vessels in the image, what fraction did our model find?"
    In clinical settings this is CRITICAL — a missed vessel could mean
    missing a hemorrhage or microaneurysm.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fn = (target_flat * (1 - pred_flat)).sum()
    return (tp + smooth) / (tp + fn + smooth)


def specificity(pred, target, smooth=1e-6):
    """
    Specificity (True Negative Rate)
    = TN / (TN + FP)

    "Of all background pixels, how many did we correctly leave as background?"
    High specificity means fewer false alarms.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tn = ((1 - pred_flat) * (1 - target_flat)).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tn + smooth) / (tn + fp + smooth)


def precision_score(pred, target, smooth=1e-6):
    """
    Precision (Positive Predictive Value)
    = TP / (TP + FP)

    "Of everything our model labeled as vessel, how much actually was vessel?"
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tp + smooth) / (tp + fp + smooth)


def pixel_accuracy(pred, target):
    """
    Standard pixel-wise accuracy. Included for completeness but NOT
    reliable for segmentation due to class imbalance.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()
    correct = (pred_flat == target_flat).sum()
    total = target_flat.numel()
    return correct.float() / total


# ============================================================
# Threshold-Independent Metrics (operate on probability maps)
# ============================================================

def compute_auc_roc(pred_probs, targets):
    """
    AUC-ROC: Area Under the Receiver Operating Characteristic Curve.

    Measures overall discriminative ability across ALL possible thresholds.
    A value of 0.5 means random guessing; 1.0 means perfect separation.

    Args:
        pred_probs: Raw sigmoid probabilities (numpy, flattened)
        targets: Binary ground truth (numpy, flattened)
    """
    try:
        return roc_auc_score(targets, pred_probs)
    except ValueError:
        return 0.0


def compute_auc_pr(pred_probs, targets):
    """
    AUC-PR: Area Under the Precision-Recall Curve.

    More informative than AUC-ROC for imbalanced datasets (which vessel
    segmentation always is — vessels are ~10% of pixels).
    """
    try:
        return average_precision_score(targets, pred_probs)
    except ValueError:
        return 0.0


# ============================================================
# Batch Evaluation Helper
# ============================================================

def evaluate_batch(pred_logits, targets, threshold=0.5):
    """
    Compute ALL metrics for a single batch.

    Args:
        pred_logits: Raw model output BEFORE sigmoid (B, 1, H, W)
        targets: Binary ground truth (B, 1, H, W)
        threshold: Binarization threshold for the sigmoid output

    Returns:
        dict with all metric values
    """
    with torch.no_grad():
        pred_probs = torch.sigmoid(pred_logits)
        pred_binary = (pred_probs > threshold).float()

        metrics = {
            'dice': dice_coefficient(pred_binary, targets).item(),
            'iou': iou_score(pred_binary, targets).item(),
            'sensitivity': sensitivity(pred_binary, targets).item(),
            'specificity': specificity(pred_binary, targets).item(),
            'precision': precision_score(pred_binary, targets).item(),
            'accuracy': pixel_accuracy(pred_binary, targets).item(),
        }

    return metrics


def evaluate_full(pred_probs_all, targets_all, threshold=0.5):
    """
    Compute ALL metrics including AUC on the full dataset (after collecting
    all predictions across batches).

    Args:
        pred_probs_all: numpy array of all sigmoid probabilities (flattened)
        targets_all: numpy array of all binary targets (flattened)
    """
    pred_binary = (pred_probs_all > threshold).astype(np.float32)

    # Pixel-level metrics
    tp = (pred_binary * targets_all).sum()
    fp = (pred_binary * (1 - targets_all)).sum()
    fn = ((1 - pred_binary) * targets_all).sum()
    tn = ((1 - pred_binary) * (1 - targets_all)).sum()

    smooth = 1e-6
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    sens = (tp + smooth) / (tp + fn + smooth)
    spec = (tn + smooth) / (tn + fp + smooth)
    prec = (tp + smooth) / (tp + fp + smooth)
    acc = (tp + tn) / (tp + tn + fp + fn)

    # AUC metrics
    auc_roc = compute_auc_roc(pred_probs_all, targets_all)
    auc_pr = compute_auc_pr(pred_probs_all, targets_all)

    return {
        'dice': dice,
        'iou': iou,
        'sensitivity': sens,
        'specificity': spec,
        'precision': prec,
        'accuracy': acc,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
    }



# 3 from src.trainer import DiceBCELoss, train_model
"""
Retinal DR Detection - Generic Training Engine
================================================
Provides a reusable training loop with:
- Mixed precision training (faster on modern GPUs)
- Learning rate scheduling (OneCycleLR / ReduceLROnPlateau)
- Early stopping (saves GPU hours when model plateaus)
- Model checkpointing (saves best model based on val Dice)
- Metric tracking per epoch for later plotting
"""

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm




# ============================================================
# Loss Functions
# ============================================================

class DiceLoss(nn.Module):
    """
    Dice Loss = 1 - Dice Coefficient

    Directly optimizes the Dice score, which is our primary evaluation
    metric. Works much better than BCE alone for imbalanced segmentation.
    """
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred_logits, targets):
        pred = torch.sigmoid(pred_logits)
        pred_flat = pred.contiguous().view(-1)
        target_flat = targets.contiguous().view(-1)

        intersection = (pred_flat * target_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )
        return 1.0 - dice


class DiceBCELoss(nn.Module):
    """
    Combined Dice Loss + Binary Cross Entropy Loss.

    Why combine?
    - BCE provides stable gradients at the pixel level (good for learning)
    - Dice directly optimizes our evaluation metric (good for performance)
    - Together they converge faster and reach better optima.

    This is the most commonly used loss for medical image segmentation.
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.dice_loss = DiceLoss(smooth)
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, pred_logits, targets):
        dice = self.dice_loss(pred_logits, targets)
        bce = self.bce_loss(pred_logits, targets)
        return self.dice_weight * dice + self.bce_weight * bce


# ============================================================
# Training & Validation Steps
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    """
    Train for one epoch.

    Args:
        model: Segmentation model
        dataloader: Training DataLoader
        optimizer: Adam/AdamW optimizer
        criterion: Loss function (DiceBCELoss)
        device: cuda/cpu
        scaler: GradScaler for mixed precision (optional)

    Returns:
        dict with average loss and metrics for this epoch
    """
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    for images, masks in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        # Track metrics
        batch_metrics = evaluate_batch(outputs, masks)
        epoch_loss += loss.item()
        for k in epoch_metrics:
            epoch_metrics[k] += batch_metrics[k]
        num_batches += 1

    # Average over batches
    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


def validate(model, dataloader, criterion, device):
    """
    Validate the model (no gradient computation).

    Returns:
        dict with average loss and metrics for validation set
    """
    model.eval()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            batch_metrics = evaluate_batch(outputs, masks)
            epoch_loss += loss.item()
            for k in epoch_metrics:
                epoch_metrics[k] += batch_metrics[k]
            num_batches += 1

    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


# ============================================================
# Full Training Loop
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    num_epochs=50,
    scheduler=None,
    early_stopping_patience=10,
    save_dir='checkpoints',
    model_name='model',
    use_amp=True,
):
    """
    Complete training loop with early stopping and checkpointing.

    Args:
        model: Segmentation model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        optimizer: Optimizer (e.g., AdamW)
        criterion: Loss function
        device: 'cuda' or 'cpu'
        num_epochs: Maximum number of epochs
        scheduler: LR scheduler (optional)
        early_stopping_patience: Stop if val Dice doesn't improve for N epochs
        save_dir: Directory to save model checkpoints
        model_name: Name prefix for saved checkpoint files
        use_amp: Whether to use automatic mixed precision

    Returns:
        history: dict with lists of train/val metrics per epoch
        best_model_path: path to the best saved model
    """
    os.makedirs(save_dir, exist_ok=True)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None

    history = {
        'train_loss': [], 'val_loss': [],
        'train_dice': [], 'val_dice': [],
        'train_iou': [], 'val_iou': [],
        'train_sensitivity': [], 'val_sensitivity': [],
        'train_specificity': [], 'val_specificity': [],
    }

    best_val_dice = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_path = os.path.join(save_dir, f'{model_name}_best.pth')

    print(f"\n{'='*60}")
    print(f"Training {model_name} for up to {num_epochs} epochs")
    print(f"Device: {device} | AMP: {use_amp} | Patience: {early_stopping_patience}")
    print(f"{'='*60}\n")

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        # Train
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)

        # Validate
        val_metrics = validate(model, val_loader, criterion, device)

        # Update scheduler
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_metrics['dice'])
            else:
                scheduler.step()

        # Record history
        for key in ['loss', 'dice', 'iou', 'sensitivity', 'specificity']:
            history[f'train_{key}'].append(train_metrics[key])
            history[f'val_{key}'].append(val_metrics[key])

        elapsed = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']

        # Print epoch summary
        print(
            f"Epoch {epoch:3d}/{num_epochs} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Dice: {val_metrics['dice']:.4f} | "
            f"Val IoU: {val_metrics['iou']:.4f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
        )

        # Check for improvement
        if val_metrics['dice'] > best_val_dice:
            best_val_dice = val_metrics['dice']
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': best_val_dice,
                'history': history,
            }, best_model_path)
            print(f"  ★ New best model saved! Dice: {best_val_dice:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"\n⏹ Early stopping at epoch {epoch}. "
                      f"Best Dice: {best_val_dice:.4f} at epoch {best_epoch}")
                break

    print(f"\n{'='*60}")
    print(f"Training complete. Best Val Dice: {best_val_dice:.4f} (epoch {best_epoch})")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*60}")

    return history, best_model_path


# 4 from src.visualize import plot_training_history, plot_predictions, plot_roc_pr_curves

"""
Retinal DR Detection - Visualization Utilities
================================================
Provides functions for:
- Training history curves (loss, Dice, IoU over epochs)
- Prediction overlays (original image + ground truth + model prediction)
- Side-by-side model comparison charts
- Confusion matrix display
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, precision_recall_curve


# ============================================================
# Training History Plots
# ============================================================

def plot_training_history(history, model_name='Model'):
    """
    Plot loss and all key metrics over training epochs.

    Creates a 2x2 grid:
    - Top-left: Loss (train vs val)
    - Top-right: Dice (train vs val)
    - Bottom-left: IoU (train vs val)
    - Bottom-right: Sensitivity & Specificity (val only)
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{model_name} — Training History', fontsize=16, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val')
    axes[0, 0].set_title('Loss (DiceBCE)')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Dice
    axes[0, 1].plot(epochs, history['train_dice'], 'b-', label='Train')
    axes[0, 1].plot(epochs, history['val_dice'], 'r-', label='Val')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # IoU
    axes[1, 0].plot(epochs, history['train_iou'], 'b-', label='Train')
    axes[1, 0].plot(epochs, history['val_iou'], 'r-', label='Val')
    axes[1, 0].set_title('IoU (Jaccard)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Sensitivity & Specificity
    axes[1, 1].plot(epochs, history['val_sensitivity'], 'g-', label='Val Sensitivity')
    axes[1, 1].plot(epochs, history['val_specificity'], 'm-', label='Val Specificity')
    axes[1, 1].set_title('Sensitivity & Specificity (Val)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Prediction Visualization
# ============================================================

def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return torch.clamp(tensor, 0, 1).permute(1, 2, 0).numpy()


def plot_predictions(model, dataloader, device, num_samples=4, threshold=0.5, model_name='Model'):
    """
    Visualize model predictions on validation/test data.

    For each sample shows 4 columns:
    1. Original fundus image
    2. Ground truth vessel mask
    3. Model's predicted mask
    4. Overlay (original + prediction in red + ground truth in green)
    """
    model.eval()
    images_shown = 0

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    fig.suptitle(f'{model_name} — Predictions', fontsize=16, fontweight='bold')

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = ['Original Image', 'Ground Truth', 'Prediction', 'Overlay']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=12, fontweight='bold')

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            preds_binary = (preds > threshold).float()

            for i in range(images.shape[0]):
                if images_shown >= num_samples:
                    break

                img_np = denormalize(images[i])
                gt_np = masks[i].squeeze().cpu().numpy()
                pred_np = preds_binary[i].squeeze().cpu().numpy()

                # Original image
                axes[images_shown, 0].imshow(img_np)
                axes[images_shown, 0].axis('off')

                # Ground truth
                axes[images_shown, 1].imshow(gt_np, cmap='gray')
                axes[images_shown, 1].axis('off')

                # Prediction
                axes[images_shown, 2].imshow(pred_np, cmap='gray')
                axes[images_shown, 2].axis('off')

                # Overlay: Green = GT, Red = Prediction
                overlay = img_np.copy()
                overlay[gt_np > 0.5] = [0, 1, 0]      # Green for ground truth
                overlay[pred_np > 0.5] = [1, 0, 0]     # Red for prediction
                # Yellow where both overlap (correct predictions)
                both = (gt_np > 0.5) & (pred_np > 0.5)
                overlay[both] = [1, 1, 0]
                axes[images_shown, 3].imshow(overlay)
                axes[images_shown, 3].axis('off')

                images_shown += 1

            if images_shown >= num_samples:
                break

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# ROC and PR Curve Plots
# ============================================================

def plot_roc_pr_curves(pred_probs, targets, model_name='Model'):
    """
    Plot ROC curve and Precision-Recall curve side by side.

    Args:
        pred_probs: Flattened numpy array of sigmoid probabilities
        targets: Flattened numpy array of binary ground truth
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — ROC & PR Curves', fontsize=14, fontweight='bold')

    # ROC Curve
    fpr, tpr, _ = roc_curve(targets, pred_probs)
    ax1.plot(fpr, tpr, 'b-', linewidth=2)
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.grid(True, alpha=0.3)

    # PR Curve
    precision, recall, _ = precision_recall_curve(targets, pred_probs)
    ax2.plot(recall, precision, 'r-', linewidth=2)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Model Comparison Table
# ============================================================

def print_comparison_table(results_dict):
    """
    Print a formatted comparison table of all models.

    Args:
        results_dict: dict of {model_name: {metric_name: value}}

    Example:
        results = {
            'U-Net': {'dice': 0.82, 'iou': 0.70, ...},
            'TransUNet': {'dice': 0.85, 'iou': 0.74, ...},
        }
    """
    metrics = ['dice', 'iou', 'sensitivity', 'specificity', 'precision', 'accuracy', 'auc_roc', 'auc_pr']
    header = f"{'Model':<20}" + "".join([f"{m:>14}" for m in metrics])
    print("=" * len(header))
    print(header)
    print("=" * len(header))

    for model_name, scores in results_dict.items():
        row = f"{model_name:<20}"
        for m in metrics:
            val = scores.get(m, 0)
            row += f"{val:>14.4f}"
        print(row)

    print("=" * len(header))

    # Highlight best model per metric
    print("\n★ Best per metric:")
    for m in metrics:
        best_model = max(results_dict, key=lambda x: results_dict[x].get(m, 0))
        best_val = results_dict[best_model].get(m, 0)
        print(f"  {m:>14}: {best_model} ({best_val:.4f})")


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
import zipfile


zip_path = "DR_dataset_processed.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
DATA_DIR = 'dataset'
TRAIN_CSV = os.path.join(DATA_DIR, 'train_split.csv')
VAL_CSV = os.path.join(DATA_DIR, 'val_split.csv')
TEST_CSV = os.path.join(DATA_DIR, 'test_split.csv')

IMG_SIZE = 512
BATCH_SIZE = 4    # Transformer + CNN = high memory
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5
PATIENCE = 10
MODEL_NAME = 'transunet'

## TransUNet Architecture Implementation

In [ ]:
class TransformerBlock(nn.Module):
    """
    Standard Transformer Encoder Block:
    Multi-Head Self-Attention + Feed-Forward Network with Layer Norm.
    """
    def __init__(self, dim, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Self-attention with residual
        normed = self.norm1(x)
        x = x + self.attn(normed, normed, normed)[0]
        # FFN with residual
        x = x + self.mlp(self.norm2(x))
        return x


class DecoderBlock(nn.Module):
    """Upsample + concat skip + double convolution."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class TransUNet(nn.Module):
    """
    TransUNet: CNN Encoder (ResNet50) → Transformer → CNN Decoder.

    The CNN encoder extracts local features at multiple scales.
    The Transformer processes the bottleneck features for global context.
    The CNN decoder upsamples with skip connections for spatial precision.
    """
    def __init__(self, in_channels=3, out_channels=1, img_size=512,
                 embed_dim=768, num_heads=12, num_layers=12, pretrained=True):
        super().__init__()
        self.img_size = img_size

        # === CNN Encoder (ResNet50, first 3 stages) ===
        resnet = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 256 ch, /2
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)           # 256 ch, /4
        self.enc3 = resnet.layer2                                          # 512 ch, /8
        self.enc4 = resnet.layer3                                          # 1024 ch, /16

        # === Projection to Transformer dimension ===
        self.proj = nn.Conv2d(1024, embed_dim, 1)

        # Feature map size at bottleneck
        self.feat_size = img_size // 16  # e.g., 512/16 = 32
        num_patches = self.feat_size ** 2  # 32×32 = 1024 tokens

        # Positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # === Transformer Encoder ===
        self.transformer = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(embed_dim)

        # === CNN Decoder ===
        self.dec4 = DecoderBlock(embed_dim, 512, 256)
        self.dec3 = DecoderBlock(256, 256, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 0, 32)  # No skip for final level

        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        # === CNN Encoding ===
        s1 = self.enc1(x)       # (B, 64, H/2, W/2)
        s2 = self.enc2(s1)      # (B, 256, H/4, W/4)
        s3 = self.enc3(s2)      # (B, 512, H/8, W/8)
        s4 = self.enc4(s3)      # (B, 1024, H/16, W/16)

        # === Projection + Transformer ===
        t = self.proj(s4)                          # (B, 768, H/16, W/16)
        B, C, H, W = t.shape
        t = t.flatten(2).transpose(1, 2)           # (B, H*W, 768)
        t = t + self.pos_embed[:, :t.size(1), :]   # Add positional encoding
        t = self.transformer(t)                    # Self-attention magic
        t = self.norm(t)
        t = t.transpose(1, 2).view(B, C, H, W)    # Back to 2D: (B, 768, H/16, W/16)

        # === CNN Decoding ===
        d4 = self.dec4(t, s3)       # Use encoder skip from stage 3
        d3 = self.dec3(d4, s2)      # Skip from stage 2
        d2 = self.dec2(d3, s1)      # Skip from stage 1
        d1 = self.dec1(d2)          # Final upsample (no skip)

        out = F.interpolate(d1, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return self.final(out)

In [ ]:
train_dataset = RetinalDataset(TRAIN_CSV, DATA_DIR, get_train_transforms(IMG_SIZE))
val_dataset = RetinalDataset(VAL_CSV, DATA_DIR, get_val_transforms(IMG_SIZE))
test_dataset = RetinalDataset(TEST_CSV, DATA_DIR, get_val_transforms(IMG_SIZE))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# NOTE: Using fewer transformer layers (6 instead of 12) for T4 GPU memory
model = TransUNet(in_channels=3, out_channels=1, img_size=IMG_SIZE,
                  embed_dim=768, num_heads=12, num_layers=6, pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
import torch
import gc

gc.collect()                  # clears Python garbage
torch.cuda.empty_cache()      # releases unused GPU memory

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = DiceBCELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

history, best_model_path = train_model(
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, criterion=criterion, device=device,
    num_epochs=NUM_EPOCHS, scheduler=scheduler, early_stopping_patience=PATIENCE,
    save_dir='checkpoints', model_name=MODEL_NAME, use_amp=True,
)

In [ ]:
plot_training_history(history, model_name='TransUNet')

checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images))
        all_preds.append(probs.cpu().numpy().flatten())
        all_targets.append(masks.numpy().flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
test_results = evaluate_full(all_preds, all_targets)

print('\nTEST RESULTS — TransUNet')
for k, v in test_results.items():
    print(f'  {k:>14}: {v:.4f}')

In [ ]:
plot_predictions(model, test_loader, device, num_samples=4, model_name='TransUNet')
plot_roc_pr_curves(all_preds, all_targets, model_name='TransUNet')



In [ ]:
import json
SAVE_DIR = 'results'
os.makedirs('results', exist_ok=True)
with open(f'results/{MODEL_NAME}_results.json', 'w') as f:
    json.dump(test_results, f, indent=2, default=str)

print(f'Results saved to results/{MODEL_NAME}_results.json')

In [ ]:
# 2. Save Training History plot
fig_hist = plot_training_history(history, model_name=MODEL_NAME)
fig_hist.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_training_history.png'), dpi=150, bbox_inches='tight')
plt.close(fig_hist)
print(f'✓ Training history saved')
# 3. Save Prediction Overlays
fig_pred = plot_predictions(model, test_loader, device, num_samples=4, model_name=MODEL_NAME)
fig_pred.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_predictions.png'), dpi=150, bbox_inches='tight')
plt.close(fig_pred)
print(f'✓ Predictions saved')
# 4. Save ROC & PR Curves
fig_roc = plot_roc_pr_curves(all_preds, all_targets, model_name=MODEL_NAME)
fig_roc.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_roc_pr.png'), dpi=150, bbox_inches='tight')
plt.close(fig_roc)
print(f'✓ ROC/PR curves saved')

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def denormalize_image(img_tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """
    Convert normalized tensor [C,H,W] to displayable numpy image [H,W,C].
    """
    img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)  # C,H,W -> H,W,C
    mean = np.array(mean)
    std = np.array(std)
    img = np.clip(img * std + mean, 0, 1)
    return img


@torch.no_grad()
def visualize_predictions(
    model,
    dataloader,
    device,
    num_samples=8,
    threshold=0.5,
    save_dir='predictions',
    model_name='Model',
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
    apply_sigmoid=True
):
    """
    Visualize model predictions with 5 panels per sample:
      1. Original fundus image
      2. Ground truth vessel mask
      3. Prediction probability heatmap
      4. Binary prediction overlay on original image
      5. Error map: green=TP, red=FP, blue=FN

    Parameters
    ----------
    model : torch.nn.Module
        Trained segmentation model.
    dataloader : DataLoader
        Validation or test dataloader.
    device : torch.device
        CPU or CUDA device.
    num_samples : int
        Number of samples to visualize.
    threshold : float
        Threshold for binary mask conversion.
    save_dir : str
        Folder to save prediction figures.
    model_name : str
        Name shown in plot title.
    mean, std : tuple
        Normalization stats used for input images.
    apply_sigmoid : bool
        Set True if model outputs logits.
        Set False if model already outputs probabilities.
    """
    model.eval()
    os.makedirs(save_dir, exist_ok=True)

    shown = 0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        if apply_sigmoid:
            probs = torch.sigmoid(outputs)
        else:
            probs = outputs

        probs_np = probs.detach().cpu().numpy()
        masks_np = masks.detach().cpu().numpy()

        for i in range(images.shape[0]):
            if shown >= num_samples:
                break

            # Original image
            img = denormalize_image(images[i], mean=mean, std=std)

            # Ground truth and prediction
            gt = masks_np[i].squeeze()
            prob = probs_np[i].squeeze()
            pred = (prob > threshold).astype(np.float32)

            # Overlay: red predicted vessels on original image
            overlay = img.copy()
            overlay[pred > 0.5] = [1.0, 0.15, 0.15]

            # Error map
            err = np.zeros((*gt.shape, 3), dtype=np.float32)
            err[..., 1] = pred * gt              # TP = green
            err[..., 0] = pred * (1 - gt)        # FP = red
            err[..., 2] = (1 - pred) * gt        # FN = blue

            # Per-image metrics
            inter = (pred * gt).sum()
            img_dice = (2 * inter + 1e-6) / (pred.sum() + gt.sum() + 1e-6)
            img_sens = inter / (gt.sum() + 1e-6)
            vessel_pct = gt.mean() * 100

            # Create figure
            fig, axes = plt.subplots(1, 5, figsize=(25, 5))
            fig.suptitle(
                f'{model_name} — Sample {shown+1} | Dice: {img_dice:.4f} | '
                f'Sensitivity: {img_sens:.4f} | Vessel pixels: {vessel_pct:.2f}%',
                fontsize=12, fontweight='bold'
            )

            # 1. Original image
            axes[0].imshow(img)
            axes[0].set_title('Original Image', fontsize=10)
            axes[0].axis('off')

            # 2. Ground truth
            axes[1].imshow(gt, cmap='gray')
            axes[1].set_title('Ground Truth', fontsize=10)
            axes[1].axis('off')

            # 3. Probability heatmap
            im = axes[2].imshow(prob, cmap='hot', vmin=0, vmax=1)
            axes[2].set_title('Prediction Probability', fontsize=10)
            axes[2].axis('off')
            plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

            # 4. Overlay
            axes[3].imshow(overlay)
            axes[3].set_title('Prediction Overlay', fontsize=10)
            axes[3].axis('off')

            # 5. Error map
            axes[4].imshow(err)
            axes[4].set_title('Error Map', fontsize=10)
            axes[4].axis('off')

            tp_patch = mpatches.Patch(color='green', label='TP (correct)')
            fp_patch = mpatches.Patch(color='red', label='FP (false alarm)')
            fn_patch = mpatches.Patch(color='blue', label='FN (missed)')
            axes[4].legend(handles=[tp_patch, fp_patch, fn_patch],
                           loc='lower right', fontsize=8)

            plt.tight_layout()

            # Save each figure
            out_path = os.path.join(save_dir, f'prediction_{shown+1:02d}.png')
            plt.savefig(out_path, dpi=120, bbox_inches='tight')
            plt.show()
            plt.close(fig)

            shown += 1

        if shown >= num_samples:
            break

    print(f'{shown} prediction visualizations saved to: {save_dir}')

In [ ]:

visualize_predictions(
    model=model,
    dataloader=test_loader,
    device=device,
    num_samples=8,
    threshold=0.5,
    save_dir='predictions_TransUNet',
    model_name='TransUNet'
)

In [ ]:
import shutil

folder_path = "results"   # <-- your folder name
output_zip = "UNet_resnet34_results.zip"

shutil.make_archive("results", 'zip', folder_path)

print("ZIP created:", output_zip)

In [ ]:
import shutil

folder_path = "predictions_TransUNet"   # <-- your folder name
output_zip = "predictions_TransUNet.zip"

shutil.make_archive("predictions_TransUNet", 'zip', folder_path)

print("ZIP created:", output_zip)